## Instruction

this code extracts data from the zip file. the zip file and this code file must be in the same directory.

- Before executing this code, download the ZIP file **FFAEXT.ZIP**.
- Move the ZIP file to the same folder as the code file (**.py** / **.ipynb**).
- Finally, run the code.
- the output excel file is in the form : **FFAEXT_date hour.xlsx**

- New function
Add PDF output @ Jingyi Wu

### v2.0 changes (16/07/2026) @ Jingyi Wu

Re-parsed the fixed-width FFAEXT layout with the correct column boundaries (verified against the July 2026 extract). Fixes:

- **Address_1**: was built from wrong columns (144:206) then patched by splitting on the last `N`/`X` character — only ~9% of addresses came out clean (junk `05 01` permission-code prefixes, truncation at N/X, fused words). Now sliced directly at the real field (156:206).
- **Zip / Phone**: postcode was taken as "everything at 206+ minus the last 3 chars" — on 40% of firms the FCA repeats the company number after the dialing code, so Zip came out like `N3  1LD+44            12568` and Phone got a bogus 3-digit prefix. Now postcode = cols 219:226, dialing code = cols 226:241.
- **Name**: field extends to col 149, not 69 — 2,394 long names were truncated.
- **Address_2**: now also includes the two FF02 address fields (cols 19:69 and 69:119) that were ignored before (66k firms had data there).
- Decode with `latin-1` instead of `utf-8 errors='ignore'` so a non-ASCII byte can never shift the fixed-width columns.


In [1]:
#------------------------------------------------ Ensure required libs are installed ----------------------------------------
import importlib
import subprocess
import sys

def ensure_pip_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

# reportlab package provides reportlab.lib and reportlab.pdfgen
ensure_pip_package("reportlab", "reportlab")



#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from time import sleep
import datetime
from pandas import ExcelWriter
import os
from zipfile import ZipFile
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'GB FCAUK' ## change to current controller name

print(f"[INFO] : Running {regulatorName} Web Scraping Tool v.2.0 | Last update : 16/07/2026")
now=datetime.datetime.now()
filename = '{}_{}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])
try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment
os.chdir(scriptfolder)
writer = ExcelWriter(filename, engine='openpyxl')

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

RegType = {
"Authorised"                : "Authorised",
"No longer authorised"      : "No longer authorised" ,
"No longer registered as a" : "No longer registered as an appointed Representative" ,
"Authorised - applied to c" : "Authorised - applied to cancel" ,
"Revoked"                   : "Revoked" ,
"Authorised - in special a" : "Authorised - in special administration" ,
"Appointed representative"  : "Appointed representative" ,
"Authorised - in liquidati" : "Authorised - in liquidation" ,
"Cancelled"                 : "Cancelled" ,
"EEA Authorised - Contract" : "EEA Authorised - Contractual run-off" ,
"EEA Authorised - Supervis" : "EEA Authorised - Supervised run-off" ,
"Authorised - in administr" : "Authorised - in administration" ,
"Authorised Schedule 5 - O" : "Authorised Schedule 5 - Operator/depositary/trustee of a temporary recognised scheme" ,
"Registered"                : "Registered" ,
"EEA Authorised - Former p" : "EEA Authorised - Former passporting firm" ,
"EEA Authorised"            : "EEA Authorised" ,
"EEA Authorised - Applied"  : "EEA Authorised - applied to cancel" ,

"Authorised Schedule 5"     : "Authorised Schedule 5 - Operator/depositary/truste" ,

"Authorised - Closed to ne" : "Authorised - Closed to new business" ,
"Lapsed"                    : "Lapsed" ,
"Temporary Permission"      : "Temporary Permission" ,
"Registered - Former"       : "Registered - Former" ,
"Certified"                 : "Certified" ,
"Authorised - Closed to Re" : "Authorised - Closed to Regulated Business" ,
}
reg = 'GB FCAUK 1'
RegulationType_Revoked = ['No longer authorised', 'No longer registered as an appointed Representative', 'Revoked', 'Appointed representative', 'Cancelled', 'Lapsed']
processdate = now.strftime('%Y-%m-%d')
#------------------------------------------------ Begin_Fouction ----------------------------------------
def extract_zipFile(zipfile, select_file, dst):
    with ZipFile(zipfile, 'r') as zipObj:
        FileNames = zipObj.namelist()
        for fileName in FileNames:
            if fileName == select_file:
                zipObj.extract(fileName, dst)

def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

def join_parts(*parts):
    return ', '.join(p for p in parts if p)
#------------------------------------------------ Begin_Main ---------------------------------------- 
file = os.path.join(scriptfolder, list(filter(lambda x:x.endswith(".ZIP"), os.listdir(scriptfolder)))[0] )
filePath = os.path.join(scriptfolder, file)
print(f'[INFO] : Extraction "FFAEXT" ')
extract_zipFile(filePath, "FFAEXT", scriptfolder)
csv_file = os.path.join(scriptfolder, list(filter(lambda x:x.endswith("FFAEXT"), os.listdir(scriptfolder)))[0] )

# FFAEXT is a fixed-width file, 3 lines per firm (verified against the July 2026 extract):
#   FF01: 4:12 company number | 12:18 firm ref number | 19:149 name | 149:154 permission codes
#         | 155 flag | 156:206 address line 1 | 206:256 address line 2
#   FF02: 19:69 address line 3 | 69:119 address line 4 | 119:169 town/city | 169:219 county
#         | 219:226 postcode | 226:241 dialing code | 241+ company number repeated
#   FF03: 19:69 phone | 94:144 regulation status | 144+ dates
# latin-1 maps every byte to exactly one char, so column positions can never shift
with open(csv_file, "rb") as myfile:
    print(f'[INFO] : Data structuring "FFAEXT" ...')
    lines = [line.decode('latin-1').rstrip('\r\n') for line in myfile]

if len(lines) % 3 != 0:
    raise ValueError(f'FFAEXT has {len(lines)} lines, not a multiple of 3 - file layout changed')

print(f'[INFO] : wait, {len(lines)//3} firms processing ...')
for i in range(0, len(lines), 3):
    l1, l2, l3 = lines[i], lines[i+1], lines[i+2]
    if l1[:4] != 'FF01' or l2[:4] != 'FF02' or l3[:4] != 'FF03':
        raise ValueError(f'unexpected record types at line {i+1} - file layout changed')

    sqldict['Name'].append(l1[19:149].strip())
    sqldict['InternalID_1_type'].append('Registered company number')
    sqldict['InternalID_1'].append(l1[4:12].strip())
    sqldict['InternalID_2_type'].append('Firm reference number')
    sqldict['InternalID_2'].append(l1[12:18].strip())

    rt_raw = l3[94:119].strip()
    sqldict['RegulationType'].append(RegType.get(rt_raw, rt_raw))

    sqldict['Address_1'].append(l1[156:206].strip())
    sqldict['Address_2'].append(join_parts(l1[206:256].strip(), l2[19:69].strip(), l2[69:119].strip(), l2[169:219].strip()))
    sqldict['City'].append(l2[119:169].strip())
    sqldict['Zip'].append(' '.join(l2[219:226].split()))

    dial = l2[226:241].strip().rstrip('A') # source data sometimes appends a stray 'A' to the dialing code
    phone = l3[19:69].strip()
    sqldict['Phone'].append((dial + ' ' + phone).strip() if phone else '')

    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegCtry'].append(reg.split(' ')[0])
    sqldict['RegCode'].append(reg.split(' ')[1])
    sqldict['ListCode'].append(reg.split(' ')[-1])

sqldict = bourange_same_length_array(sqldict)
os.remove(csv_file)
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df_sql = pd.DataFrame(sqldict)

# drop duplicates and Revoked company
for Reg_Revoked in RegulationType_Revoked :
    df_sql =  df_sql.drop(df_sql[df_sql.RegulationType == Reg_Revoked].index) 

# df_sql = df_sql.drop_duplicates(subset = ["Name", "Address_1"], keep = 'first')
df_sql = df_sql.reset_index(drop=True)

print(f'[INFO] : wait, save data ({len(df_sql)} company) to excel file ...')

# Split DataFrame into chunks and save each chunk to a separate sheet
chunk_size = 1000000  # Adjust chunk size as needed
num_chunks = len(df_sql) // chunk_size + 1

for i in range(num_chunks):
    start_row = i * chunk_size
    end_row = (i + 1) * chunk_size
    chunk_df = df_sql.iloc[start_row:end_row]
    sheet_name = f'SQL Ready Part {i+1}'
    chunk_df.to_excel(writer, sheet_name=sheet_name, index=False)

writer.close()
sleep(2)
print('[INFO] : Finish | OK')

print('[INFO] : PDF generation')
def export_list_to_pdf(data_list, pdf_filename):
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    c.setFont("Helvetica", 12)
    x = 72
    y = 720
    max_lines_per_page = 33 # Maximum number of lines per page
    line_count = 0
    
    # Add each item from the list to the PDF
    for item in data_list:
        c.drawString(x, y, str(item))
        y -= 20  # Move to the next line
        line_count += 1
        if line_count >= max_lines_per_page: # Check if we need to add a new page
            c.showPage()  # Add a new page
            c.setFont("Helvetica", 12)  # Reset font
            x = 72  # Reset x position
            y = 720  # Reset y position
            line_count = 0  # Reset line counter
            
    c.save()
regulatorName = 'GB_FCAUK_FFAXT_List_1 '
filename = '{} Ready {}'.format(regulatorName, str(now).replace(":",".")[:-7])
export_list_to_pdf(df_sql['Name'].tolist(), f'{filename}.pdf')

[INFO] : Running GB FCAUK Web Scraping Tool v.2.0 | Last update : 16/07/2026
[INFO] : Extraction "FFAEXT" 
[INFO] : Data structuring "FFAEXT" ...
[INFO] : wait, 288379 firms processing ...
[INFO] : wait, save data (36288 company) to excel file ...
[INFO] : Finish | OK
[INFO] : PDF generation
